Import the Mlp class and the Protein Strings

In [1]:
import torch 
from torch import nn
import torch.nn.functional as F
from torch import optim
from protgen.models.mlp import MLP
from protgen.data import ProteinDataset
from protgen.data.make_tokens import TokenDataset
from torch.utils.data import DataLoader
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens

Load the Data 

In [2]:
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get train tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")

#Get validation tensors
X_val = val_data.tokens
y_val_labels = val_data.labels 
y_val_shifted = val_data.tokens_shifted

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])


Create the model and test it with a small input 

In [3]:
mlp = MLP()

X = torch.LongTensor([[0, 0, 1], [0, 1, 2], [1, 2, 3]])
Y = torch.LongTensor([1,2,3])

logits = mlp(X)
targets = Y

Loss = F.cross_entropy(logits, targets)
print(f"Logits: {logits}")
print(f"Targets: {targets}")
print(f"Loss: {Loss}")


Logits: tensor([[-0.1223, -0.0141, -0.0333,  0.1124,  0.1022,  0.0868, -0.0842,  0.0428,
         -0.0616,  0.0253, -0.1504,  0.1208, -0.0378,  0.1544,  0.0097, -0.0178,
          0.1603,  0.0358, -0.0332,  0.0046, -0.0927, -0.1012, -0.0424],
        [-0.0326, -0.0206,  0.0075,  0.1755,  0.1419,  0.1077, -0.0180, -0.0030,
         -0.1530, -0.1420, -0.1236,  0.1509,  0.0837,  0.2006, -0.1198,  0.1097,
          0.0969,  0.2830, -0.0144, -0.1130, -0.0731, -0.1654, -0.2366],
        [-0.2570, -0.0674, -0.0245,  0.0522,  0.1397,  0.0274, -0.2184, -0.0409,
          0.0360,  0.1587, -0.2163, -0.0089, -0.0138,  0.0909,  0.0313, -0.0674,
          0.1472,  0.0510,  0.1065, -0.0257, -0.0068, -0.0160, -0.0544]],
       grad_fn=<AddmmBackward0>)
Targets: tensor([1, 2, 3])
Loss: 3.1268727779388428


Optimizer Function

In [4]:
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

Accuracy Function

In [ ]:
def accuracy(ys, ts):
    #Finding the token with the highest probability for each sample in the batch
    predictions = torch.argmax(ys, dim=1)
    #Comparing with the target and counting the number of correct predictions
    correct_prediction = (predictions == ts).sum().item()
    # Return counter of correct predictions
    return float(correct_prediction)

Overfitting over the small input test

In [6]:
for epoch in range(1000):
    logits = mlp(X)
    loss = F.cross_entropy(logits, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

Epoch 0: Loss = 3.0940
Epoch 100: Loss = 0.0742
Epoch 200: Loss = 0.0127
Epoch 300: Loss = 0.0057
Epoch 400: Loss = 0.0033
Epoch 500: Loss = 0.0021
Epoch 600: Loss = 0.0015
Epoch 700: Loss = 0.0011
Epoch 800: Loss = 0.0009
Epoch 900: Loss = 0.0007


(OUTDATED!) Format the tranning data and create the 3 Tokens sequences test 

In [44]:
#Add double padding in the begging of the sequence so the first input is [0, 0, 1]
X_train_pad = F.pad(X_train, (2, 0), "constant", 0)
#print(f"Padded train data shape: {X_train_pad.shape}")
#print(f"Padded train data: {X_train_pad}")
#print(f"Original shifted data shape: {y_train_shifted.shape}")  
#print(f"Shifted train data: {y_train_shifted}")

X_train_inputs = X_train_pad.unfold(dimension=1, size=3, step=1)
print(f"Unfolded train inputs shape: {X_train_inputs.shape}")
#print(f"Unfolded train inputs: {X_train_inputs}")

X_val_pad = F.pad(X_val, (2, 0), "constant", 0)
X_val_inputs = X_val_pad.unfold(dimension=1, size=3, step=1)


#for i in range(X_train_inputs.shape[1]):
#    if X_train_inputs[1, i, 2] != y_train_shifted[1, i]:
#        print(f"Target input pair at index {i}: Input: {X_train_inputs[1, i, :]}, Target: {y_train_shifted[1, i]}")

X_train_inputs = X_train_inputs.reshape(-1, 3)
y_train_shifted = y_train_shifted.reshape(-1)
X_val_inputs = X_val_inputs.reshape(-1, 3)
y_val_shifted = y_val_shifted.reshape(-1)
print(f"Reshaped train inputs shape: {X_train_inputs.shape}")
print(f"Reshaped train targets shape: {y_train_shifted.shape}")

train_Loader = DataLoader(list(zip(X_train_inputs, y_train_shifted)), batch_size=32, shuffle=True)
val_Loader = DataLoader(list(zip(X_val_inputs, y_val_shifted)), batch_size=32, shuffle=False)

Unfolded train inputs shape: torch.Size([16020, 129, 3])
Reshaped train inputs shape: torch.Size([2066580, 3])
Reshaped train targets shape: torch.Size([2066580])


Create Tokens Dataset

In [ ]:
train_dataset = TokenDataset(train_data)
val_dataset = TokenDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
train_dataset = TokenDataset(train_data)
val_dataset = TokenDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)

x_train_batch, targets_train_batch = next(iter(train_loader))
print(f"batch: x {x_train_batch.shape} {x_train_batch.dtype}, y {targets_train_batch.shape} train loader length: {train_loader.batch_size}")

batch: x torch.Size([512, 3]) torch.int64, y torch.Size([512]) train loader length: 1538048


In [ ]:
# number of training passses
num_epochs = 5
# store loss and accuracy for information
train_losses, val_losses, train_accs, val_accs = [], [], [], []

print(f"Training for {num_epochs} epochs...\n Starting Loss should be around ln(23) ≈ 3.1 for the first epoch")
# training loop
for e in range(num_epochs):
    
    #-----------------TRAINING-----------------------#

    mlp.train()  # set model to training mode

    batch_train_loss, batch_train_acc, batch_count, num_examples = 0, 0, 0, 0
    for x_train_batch, targets_train_batch in train_loader:
        # zeroize accumulated gradients in parameters
        optimizer.zero_grad()
        # predict by running forward pass
        tr_output = mlp(x_train_batch)
        # compute cross entropy loss
        tr_loss = F.cross_entropy(tr_output, targets_train_batch, ignore_index=IGNORE_INDEX)
        # compute gradients given loss
        tr_loss.backward()
        # update the parameters given the computed gradients
        optimizer.step()
        batch_train_acc += accuracy(tr_output, targets_train_batch)

        # store training batch loss
        # Cross entropy loss is averaged over the batch, so we multiply by the batch size to get the total loss for the batch
        batch_train_loss += tr_loss.item()*x_train_batch.size(0)
        batch_count += 1
        num_examples += x_train_batch.size(0)

    # store average training loss and accuracy for the epoch
    train_losses.append(batch_train_loss / (num_examples))
    train_accs.append(batch_train_acc / (num_examples))

    #-----------------VALIDATION-----------------------#

    mlp.eval()  # set model to evaluation mode
    
    with torch.no_grad():  # disable gradient computation for validation
        batch_val_loss, batch_val_acc, batch_count, num_examples = 0, 0, 0, 0
        for x_val_batch, targets_val_batch in val_loader:    
            # predict with validation input
            val_output = mlp(x_val_batch)
            # compute loss and accuracy
            batch_val_loss += F.cross_entropy(val_output, targets_val_batch, ignore_index=IGNORE_INDEX).item()*x_val_batch.size(0)
            batch_val_acc += accuracy(val_output, targets_val_batch)
            batch_count += 1
            num_examples += x_val_batch.size(0)

    # store loss and accuracy
    val_losses.append(batch_val_loss / (num_examples))
    val_accs.append(batch_val_acc / (num_examples))

    if e % 1 == 0:
        print("Epoch %i, "
              "Train Cost: %0.3f"
              "\t Train Acc: %0.3f"
              "\tVal Cost: %0.3f"
              "\t Val acc: %0.3f" % (e,
                                     train_losses[-1],
                                     train_accs[-1],
                                     val_losses[-1],
                                     val_accs[-1]))
    


Training for 5 epochs...
 Starting Loss should be around ln(23) ≈ 3.1 for the first epoch
Epoch 0, Train Cost: 2.387	 Train Acc: 0.286	Val Cost: 2.417	 Val acc: 0.277
Epoch 1, Train Cost: 2.387	 Train Acc: 0.286	Val Cost: 2.419	 Val acc: 0.271
Epoch 2, Train Cost: 2.387	 Train Acc: 0.286	Val Cost: 2.417	 Val acc: 0.275
Epoch 3, Train Cost: 2.387	 Train Acc: 0.286	Val Cost: 2.416	 Val acc: 0.277
Epoch 4, Train Cost: 2.386	 Train Acc: 0.286	Val Cost: 2.417	 Val acc: 0.271


: 